# BingeMetrics – Week 8 Power BI Export & Gold Hand-off

## Objective

Validate the approved BingeMetrics Gold tables selected for Power BI,
create controlled report-facing exports, reconcile the exports back to Gold,
and hand off the validated Gold sources to Power BI.

## Week 8 Boundary

This notebook covers:

- Gold source inventory
- Gold hand-off validation
- Export contract
- Gold profiling
- Controlled exports
- Read-back reconciliation
- Repeat-run verification
- Power BI hand-off

Power BI is allowed to consume approved Gold outputs only.

Week 9 will focus on dashboard refinement, presentation and insight storytelling.

In [0]:
catalog = "workspace.default"

gold_tables = {
    "daily_engagement": f"{catalog}.gold_daily_engagement",
    "content_performance": f"{catalog}.gold_content_performance",
    "user_engagement": f"{catalog}.gold_user_engagement"
}

gold_tables

{'daily_engagement': 'workspace.default.gold_daily_engagement',
 'content_performance': 'workspace.default.gold_content_performance',
 'user_engagement': 'workspace.default.gold_user_engagement'}

## Gold Source Register

| Gold Table | Grain | Key | Business Purpose | Power BI Use |
|---|---|---|---|---|
| gold_daily_engagement | 1 row per session_date | session_date | Daily engagement KPIs | KPI cards, trend |
| gold_content_performance | 1 row per content_id | content_id | Content performance | Content comparison |
| gold_user_engagement | 1 row per user_id | user_id | User engagement | User engagement analysis |

In [0]:
for name, table in gold_tables.items():
    print(f"\n{name}")
    print(f"Table: {table}")
    
    spark.sql(f"DESCRIBE TABLE {table}").show(truncate=False)


daily_engagement
Table: workspace.default.gold_daily_engagement
+-----------------------+-------------+-------+
|col_name               |data_type    |comment|
+-----------------------+-------------+-------+
|session_date           |date         |NULL   |
|total_sessions         |bigint       |NULL   |
|unique_users           |bigint       |NULL   |
|unique_content_items   |bigint       |NULL   |
|total_consumed_seconds |bigint       |NULL   |
|total_consumed_minutes |decimal(24,2)|NULL   |
|average_session_seconds|double       |NULL   |
|completed_sessions     |bigint       |NULL   |
|skipped_sessions       |bigint       |NULL   |
|completion_rate_percent|decimal(27,2)|NULL   |
+-----------------------+-------------+-------+


content_performance
Table: workspace.default.gold_content_performance
+-----------------------+-------------+-------+
|col_name               |data_type    |comment|
+-----------------------+-------------+-------+
|content_id             |string       |NULL   |

In [0]:
for name, table in gold_tables.items():
    count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {table}
    """).collect()[0]["row_count"]

    print(f"{name}: {count:,} rows")

daily_engagement: 91 rows
content_performance: 3,400 rows
user_engagement: 24,999 rows


In [0]:
for name, table in gold_tables.items():
    count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {table}
    """).collect()[0]["row_count"]

    print(f"{name}: {count:,} rows")

daily_engagement: 91 rows
content_performance: 3,400 rows
user_engagement: 24,999 rows


In [0]:
grain_keys = {
    "daily_engagement": "session_date",
    "content_performance": "content_id",
    "user_engagement": "user_id"
}

for name, table in gold_tables.items():
    key = grain_keys[name]

    result = spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {key}) AS distinct_keys,
            SUM(CASE WHEN {key} IS NULL THEN 1 ELSE 0 END) AS null_keys
        FROM {table}
    """).collect()[0]

    duplicate_keys = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_key_count
        FROM (
            SELECT {key}
            FROM {table}
            WHERE {key} IS NOT NULL
            GROUP BY {key}
            HAVING COUNT(*) > 1
        )
    """).collect()[0]["duplicate_key_count"]

    print(f"\n{name}")
    print(f"Key: {key}")
    print(f"Total rows: {result['total_rows']:,}")
    print(f"Distinct keys: {result['distinct_keys']:,}")
    print(f"Duplicate keys: {duplicate_keys:,}")
    print(f"Null keys: {result['null_keys']:,}")


daily_engagement
Key: session_date
Total rows: 91
Distinct keys: 91
Duplicate keys: 0
Null keys: 0

content_performance
Key: content_id
Total rows: 3,400
Distinct keys: 3,400
Duplicate keys: 0
Null keys: 0

user_engagement
Key: user_id
Total rows: 24,999
Distinct keys: 24,999
Duplicate keys: 0
Null keys: 0


In [0]:
print("=== DAILY ENGAGEMENT ===")

spark.sql(f"""
    SELECT
        MIN(session_date) AS min_date,
        MAX(session_date) AS max_date,
        SUM(total_sessions) AS total_sessions,
        SUM(unique_users) AS total_unique_users_sum,
        SUM(unique_content_items) AS total_unique_content_items_sum,
        SUM(total_consumed_minutes) AS total_consumed_minutes,
        SUM(completed_sessions) AS completed_sessions,
        SUM(skipped_sessions) AS skipped_sessions
    FROM {gold_tables["daily_engagement"]}
""").show(truncate=False)


print("=== CONTENT PERFORMANCE ===")

spark.sql(f"""
    SELECT
        COUNT(*) AS content_rows,
        SUM(total_sessions) AS total_sessions,
        SUM(unique_viewers) AS total_unique_viewers_sum,
        SUM(total_consumed_minutes) AS total_consumed_minutes,
        SUM(completed_sessions) AS completed_sessions,
        SUM(skipped_sessions) AS skipped_sessions
    FROM {gold_tables["content_performance"]}
""").show(truncate=False)


print("=== USER ENGAGEMENT ===")

spark.sql(f"""
    SELECT
        COUNT(*) AS user_rows,
        SUM(total_sessions) AS total_sessions,
        SUM(total_consumed_minutes) AS total_consumed_minutes,
        SUM(completed_sessions) AS completed_sessions,
        SUM(skipped_sessions) AS skipped_sessions
    FROM {gold_tables["user_engagement"]}
""").show(truncate=False)

=== DAILY ENGAGEMENT ===
+----------+----------+--------------+----------------------+------------------------------+----------------------+------------------+----------------+
|min_date  |max_date  |total_sessions|total_unique_users_sum|total_unique_content_items_sum|total_consumed_minutes|completed_sessions|skipped_sessions|
+----------+----------+--------------+----------------------+------------------------------+----------------------+------------------+----------------+
|2025-12-20|2026-03-31|249701        |236339                |161533                        |6241936.65            |94689             |47854           |
+----------+----------+--------------+----------------------+------------------------------+----------------------+------------------+----------------+

=== CONTENT PERFORMANCE ===
+------------+--------------+------------------------+----------------------+------------------+----------------+
|content_rows|total_sessions|total_unique_viewers_sum|total_consumed_min

In [0]:
# ============================================================
# WEEK 8 - EXPORT CONTRACT
# ============================================================

daily_export = spark.sql(f"""
    SELECT
        session_date,
        total_sessions,
        unique_users,
        unique_content_items,
        total_consumed_minutes,
        average_session_seconds,
        completed_sessions,
        skipped_sessions,
        completion_rate_percent
    FROM {gold_tables["daily_engagement"]}
    ORDER BY session_date
""")


content_export = spark.sql(f"""
    SELECT
        content_id,
        content_type,
        genre,
        language,
        is_platform_original,
        total_sessions,
        unique_viewers,
        total_consumed_minutes,
        average_session_seconds,
        completed_sessions,
        skipped_sessions,
        completion_rate_percent
    FROM {gold_tables["content_performance"]}
    ORDER BY content_id
""")


user_export = spark.sql(f"""
    SELECT
        user_id,
        total_sessions,
        total_consumed_minutes,
        completed_sessions,
        skipped_sessions,
        average_session_seconds,
        first_session_date,
        last_session_date
    FROM {gold_tables["user_engagement"]}
    ORDER BY user_id
""")


print("Export DataFrames created:")
print(f"daily_export   : {daily_export.count():,} rows")
print(f"content_export : {content_export.count():,} rows")
print(f"user_export    : {user_export.count():,} rows")

Export DataFrames created:
daily_export   : 91 rows
content_export : 3,400 rows
user_export    : 24,999 rows


In [0]:
print("=== DAILY EXPORT SCHEMA ===")
daily_export.printSchema()

print("=== CONTENT EXPORT SCHEMA ===")
content_export.printSchema()

print("=== USER EXPORT SCHEMA ===")
user_export.printSchema()

=== DAILY EXPORT SCHEMA ===
root
 |-- session_date: date (nullable = true)
 |-- total_sessions: long (nullable = true)
 |-- unique_users: long (nullable = true)
 |-- unique_content_items: long (nullable = true)
 |-- total_consumed_minutes: decimal(24,2) (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: long (nullable = true)
 |-- skipped_sessions: long (nullable = true)
 |-- completion_rate_percent: decimal(27,2) (nullable = true)

=== CONTENT EXPORT SCHEMA ===
root
 |-- content_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- language: string (nullable = true)
 |-- is_platform_original: boolean (nullable = true)
 |-- total_sessions: long (nullable = true)
 |-- unique_viewers: long (nullable = true)
 |-- total_consumed_minutes: decimal(24,2) (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: long (nullable = true)
 |-- skipped_

In [0]:
# ============================================================
# WEEK 8 - EXPORT KEY CHECK
# ============================================================

export_checks = {
    "daily_export": ("session_date", daily_export),
    "content_export": ("content_id", content_export),
    "user_export": ("user_id", user_export)
}

for name, (key, df) in export_checks.items():
    total_rows = df.count()
    distinct_keys = df.select(key).distinct().count()
    null_keys = df.filter(df[key].isNull()).count()

    print(f"\n{name}")
    print(f"Key: {key}")
    print(f"Rows: {total_rows:,}")
    print(f"Distinct keys: {distinct_keys:,}")
    print(f"Null keys: {null_keys:,}")
    print(f"Key check: {'PASS' if total_rows == distinct_keys and null_keys == 0 else 'FAIL'}")


daily_export
Key: session_date
Rows: 91
Distinct keys: 91
Null keys: 0
Key check: PASS

content_export
Key: content_id
Rows: 3,400
Distinct keys: 3,400
Null keys: 0
Key check: PASS

user_export
Key: user_id
Rows: 24,999
Distinct keys: 24,999
Null keys: 0
Key check: PASS


In [0]:
# ============================================================
# WEEK 8 - EXPORT SAMPLE CHECK
# ============================================================

print("=== DAILY EXPORT SAMPLE ===")
daily_export.show(5, truncate=False)

print("=== CONTENT EXPORT SAMPLE ===")
content_export.show(5, truncate=False)

print("=== USER EXPORT SAMPLE ===")
user_export.show(5, truncate=False)

=== DAILY EXPORT SAMPLE ===
+------------+--------------+------------+--------------------+----------------------+-----------------------+------------------+----------------+-----------------------+
|session_date|total_sessions|unique_users|unique_content_items|total_consumed_minutes|average_session_seconds|completed_sessions|skipped_sessions|completion_rate_percent|
+------------+--------------+------------+--------------------+----------------------+-----------------------+------------------+----------------+-----------------------+
|2025-12-20  |500           |492         |465                 |13006.60              |1560.79                |183               |105             |36.60                  |
|2026-01-01  |2738          |2586        |1812                |68765.95              |1506.92                |990               |576             |36.16                  |
|2026-01-02  |2774          |2633        |1777                |69452.48              |1502.22                |1072   

In [0]:
%sql
SELECT
    MIN(completion_rate_percent) AS min_completion_rate,
    MAX(completion_rate_percent) AS max_completion_rate,
    AVG(completion_rate_percent) AS avg_completion_rate
FROM workspace.default.gold_daily_engagement;

min_completion_rate,max_completion_rate,avg_completion_rate
35.43,39.64,37.911429


In [0]:
%sql
SELECT
    COUNT(*) AS invalid_rows
FROM workspace.default.gold_daily_engagement
WHERE completion_rate_percent < 0
   OR completion_rate_percent > 100
   OR total_sessions < 0
   OR unique_users < 0
   OR total_consumed_minutes < 0;

invalid_rows
0


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS distinct_keys,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM workspace.default.gold_user_engagement;

total_rows,distinct_keys,null_keys
24999,24999,0


In [0]:
print(dbutils.fs.ls("/Workspace"))

[FileInfo(path='dbfs:/Workspace/.rm-rf-guard', name='.rm-rf-guard', size=0, modificationTime=1790412988284), FileInfo(path='dbfs:/Workspace/Repos/', name='Repos/', size=4096, modificationTime=1790414595247), FileInfo(path='dbfs:/Workspace/Shared/', name='Shared/', size=4096, modificationTime=1790414595247), FileInfo(path='dbfs:/Workspace/Users/', name='Users/', size=4096, modificationTime=1790414595247)]


In [0]:
%sql
SELECT *
FROM workspace.default.gold_daily_engagement
ORDER BY session_date;

session_date,total_sessions,unique_users,unique_content_items,total_consumed_seconds,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
2025-12-20,500,492,465,780396,13006.60,1560.79,183,105,36.60
2026-01-01,2738,2586,1812,4125957,68765.95,1506.92,990,576,36.16
2026-01-02,2774,2633,1777,4167149,69452.48,1502.22,1072,538,38.64
2026-01-03,2790,2639,1791,4117139,68618.98,1475.68,1064,532,38.14
2026-01-04,2723,2571,1771,4034437,67240.62,1481.61,1008,520,37.02
2026-01-05,2693,2552,1767,4051622,67527.03,1504.5,1025,533,38.06
2026-01-06,2753,2593,1768,4167139,69452.32,1513.67,1061,507,38.54
2026-01-07,2852,2687,1818,4296223,71603.72,1506.39,1080,580,37.87
2026-01-08,2866,2712,1813,4206845,70114.08,1467.85,1080,560,37.68
2026-01-09,2800,2647,1820,4030967,67182.78,1439.63,1058,564,37.79


In [0]:
%sql
SELECT *
FROM workspace.default.gold_content_performance
ORDER BY content_id;

content_id,content_type,genre,language,is_platform_original,total_sessions,unique_viewers,total_consumed_seconds,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
C000001,VIDEO,Sports Stories,Telugu,false,98,98,137950,2299.17,1407.65,33,27,33.67
C000002,MUSIC,Jazz,Telugu,false,89,89,10480,174.67,117.75,26,21,29.21
C000003,MUSIC,Devotional,Hindi,false,75,75,12061,201.02,160.81,30,11,40.00
C000004,MUSIC,Indie Pop,Kannada,true,82,81,7618,126.97,92.9,23,24,28.05
C000005,PODCAST,Careers,Spanish,false,88,87,262991,4383.18,2988.53,33,13,37.50
C000006,VIDEO,Documentary,Hindi,false,76,76,290254,4837.57,3819.13,24,19,31.58
C000007,VIDEO,Comedy,Hindi,false,80,80,155835,2597.25,1947.94,34,13,42.50
C000008,VIDEO,Sports Stories,English,true,101,101,152999,2549.98,1514.84,46,20,45.54
C000009,VIDEO,Drama,Telugu,false,80,80,265567,4426.12,3319.59,25,18,31.25
C000010,VIDEO,Comedy,Spanish,false,100,100,158100,2635.00,1581.0,39,17,39.00


In [0]:
%sql
SELECT *
FROM workspace.default.gold_user_engagement
ORDER BY user_id;

user_id,total_sessions,total_consumed_seconds,total_consumed_minutes,completed_sessions,skipped_sessions,average_session_seconds,first_session_date,last_session_date
U000001,6,2042,34.03,2,1,340.33,2026-01-03,2026-03-30
U000002,10,14431,240.52,4,1,1443.1,2026-01-01,2026-03-29
U000003,14,27840,464.00,5,5,1988.57,2026-01-01,2026-03-25
U000004,7,7642,127.37,2,1,1091.71,2026-01-03,2026-03-16
U000005,11,12045,200.75,3,3,1095.0,2026-01-26,2026-03-27
U000006,11,23711,395.18,4,2,2155.55,2026-01-05,2026-03-28
U000007,12,17392,289.87,5,3,1449.33,2026-01-06,2026-03-18
U000008,11,15882,264.70,5,2,1443.82,2026-01-19,2026-03-25
U000009,16,14562,242.70,4,6,910.13,2026-01-01,2026-03-19
U000010,12,12728,212.13,2,4,1060.67,2026-01-07,2026-03-26


In [0]:
%sql
DESCRIBE DETAIL workspace.default.gold_daily_engagement;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,abb1398c-ad65-4533-8d13-93fcba652e13,workspace.default.gold_daily_engagement,null,,2026-09-10T14:13:08.626Z,2026-09-10T14:13:10.000Z,List(),List(),1,5845,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL workspace.default.gold_content_performance;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ef5fa9d0-8dd4-4ccc-afea-5d49d3f71da3,workspace.default.gold_content_performance,null,,2026-09-10T14:13:00.155Z,2026-09-10T14:13:02.000Z,List(),List(),1,63954,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL workspace.default.gold_user_engagement;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,51a7bd67-3721-4d32-a9c9-6ccf294d6344,workspace.default.gold_user_engagement,null,,2026-09-10T14:12:50.801Z,2026-09-10T14:12:52.000Z,List(),List(),1,386279,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants, timestampNtz)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_content_catalog,false
default,bronze_sessions,false
default,bronze_subscriptions,false
default,bronze_users,false
default,churn_risk_summary,false
default,content_performance_summary,false
default,genre_engagement_summary,false
default,gold_content,false
default,gold_content_performance,false
default,gold_content_performance_summary,false


In [0]:
gold_daily = spark.table(
    "workspace.default.gold_daily_engagement"
)

gold_content = spark.table(
    "workspace.default.gold_content_performance"
)

gold_user = spark.table(
    "workspace.default.gold_user_engagement"
)

print("Daily engagement:", gold_daily.count())
print("Content performance:", gold_content.count())
print("User engagement:", gold_user.count())

Daily engagement: 91
Content performance: 3400
User engagement: 24999


In [0]:
from pyspark.sql import functions as F

# Approved Gold sources
gold_daily = spark.table("workspace.default.gold_daily_engagement")
gold_content = spark.table("workspace.default.gold_content_performance")
gold_user = spark.table("workspace.default.gold_user_engagement")

# Controlled report-facing exports
daily_export = gold_daily.select(
    "session_date",
    "total_sessions",
    "unique_users",
    "unique_content_items",
    "total_consumed_minutes",
    "average_session_seconds",
    "completed_sessions",
    "skipped_sessions",
    "completion_rate_percent"
).orderBy("session_date")

content_export = gold_content.select(
    "content_id",
    "content_type",
    "genre",
    "language",
    "is_platform_original",
    "total_sessions",
    "unique_viewers",
    "total_consumed_minutes",
    "average_session_seconds",
    "completed_sessions",
    "skipped_sessions",
    "completion_rate_percent"
).orderBy("content_id")

user_export = gold_user.select(
    "user_id",
    "total_sessions",
    "total_consumed_minutes",
    "completed_sessions",
    "skipped_sessions",
    "average_session_seconds",
    "first_session_date",
    "last_session_date"
).orderBy("user_id")

print("Exports prepared:")
print("Daily:", daily_export.count())
print("Content:", content_export.count())
print("User:", user_export.count())

Exports prepared:
Daily: 91
Content: 3400
User: 24999


In [0]:
spark.sql("SHOW VOLUMES IN workspace.default").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|default |week4_files|
+--------+-----------+



In [0]:
# ============================================================
# WEEK 8 - EXPORT PATH TEST
# ============================================================

export_base = "/Workspace/Shared/team 4_common/data_sample/gold_exports"

print("Export path:")
print(export_base)

print("\nExisting contents:")
display(dbutils.fs.ls(export_base))

Export path:
/Workspace/Shared/team 4_common/data_sample/gold_exports

Existing contents:


path,name,size,modificationTime
dbfs:/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_content_performance/,gold_content_performance/,4096,1790414618611
dbfs:/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_daily_engagement/,gold_daily_engagement/,4096,1790414618611
dbfs:/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_user_engagement/,gold_user_engagement/,4096,1790414618611


In [0]:
# ============================================================
# WEEK 8 - CONTROLLED GOLD EXPORTS
# ============================================================

export_base = "/Workspace/Shared/team 4_common/data_sample/gold_exports"

import os

# Spark DataFrameWriter cannot write to /Workspace (WSFS) paths.
# Use toPandas().to_csv() which writes via Python file I/O on the driver.

os.makedirs(f"{export_base}/gold_daily_engagement", exist_ok=True)
daily_export.toPandas().to_csv(f"{export_base}/gold_daily_engagement/gold_daily_engagement.csv", index=False)

os.makedirs(f"{export_base}/gold_content_performance", exist_ok=True)
content_export.toPandas().to_csv(f"{export_base}/gold_content_performance/gold_content_performance.csv", index=False)

os.makedirs(f"{export_base}/gold_user_engagement", exist_ok=True)
user_export.toPandas().to_csv(f"{export_base}/gold_user_engagement/gold_user_engagement.csv", index=False)

print("Controlled Gold exports written successfully.")

Controlled Gold exports written successfully.


In [0]:
# ============================================================
# WEEK 8 - EXPORT RECONCILIATION
# ============================================================

daily_csv = f"{export_base}/gold_daily_engagement/gold_daily_engagement.csv"
content_csv = f"{export_base}/gold_content_performance/gold_content_performance.csv"
user_csv = f"{export_base}/gold_user_engagement/gold_user_engagement.csv"

# Verify export files exist
print("=== EXPORT FILE CHECK ===")
for path in [daily_csv, content_csv, user_csv]:
    print(f"{path}")
    print(f"Exists: {os.path.exists(path)}")
    print()

# Read exported CSVs back
daily_readback = spark.read.option("header", "true").option(
    "inferSchema", "true"
).csv(f"file:{daily_csv}")

content_readback = spark.read.option("header", "true").option(
    "inferSchema", "true"
).csv(f"file:{content_csv}")

user_readback = spark.read.option("header", "true").option(
    "inferSchema", "true"
).csv(f"file:{user_csv}")

# Reconciliation
print("=== EXPORT READ-BACK RECONCILIATION ===")
print(f"Daily rows:   {daily_readback.count():,}")
print(f"Content rows: {content_readback.count():,}")
print(f"User rows:    {user_readback.count():,}")

=== EXPORT FILE CHECK ===
/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_daily_engagement/gold_daily_engagement.csv
Exists: True

/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_content_performance/gold_content_performance.csv
Exists: True

/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_user_engagement/gold_user_engagement.csv
Exists: True

=== EXPORT READ-BACK RECONCILIATION ===
Daily rows:   91
Content rows: 3,400
User rows:    24,999


In [0]:
# ============================================================
# WEEK 8 - VERIFY CONTROLLED GOLD EXPORTS
# ============================================================

for path in [
    f"{export_base}/gold_daily_engagement/gold_daily_engagement.csv",
    f"{export_base}/gold_content_performance/gold_content_performance.csv",
    f"{export_base}/gold_user_engagement/gold_user_engagement.csv"
]:
    print(f"{path}")
    print(f"Exists: {os.path.exists(path)}")
    print(f"Size: {os.path.getsize(path):,} bytes")
    print()

/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_daily_engagement/gold_daily_engagement.csv
Exists: True
Size: 5,431 bytes

/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_content_performance/gold_content_performance.csv
Exists: True
Size: 226,555 bytes

/Workspace/Shared/team 4_common/data_sample/gold_exports/gold_user_engagement/gold_user_engagement.csv
Exists: True
Size: 1,277,102 bytes



In [0]:
# ============================================================
# WEEK 8 - EXPORT READ-BACK RECONCILIATION
# ============================================================

daily_csv = f"{export_base}/gold_daily_engagement/gold_daily_engagement.csv"
content_csv = f"{export_base}/gold_content_performance/gold_content_performance.csv"
user_csv = f"{export_base}/gold_user_engagement/gold_user_engagement.csv"

daily_readback = spark.read.option("header", "true").option("inferSchema", "true").csv(
    f"file:{daily_csv}"
)

content_readback = spark.read.option("header", "true").option("inferSchema", "true").csv(
    f"file:{content_csv}"
)

user_readback = spark.read.option("header", "true").option("inferSchema", "true").csv(
    f"file:{user_csv}"
)

print("=== READ-BACK ROW COUNTS ===")
print(f"Daily:    {daily_readback.count():,}")
print(f"Content:  {content_readback.count():,}")
print(f"User:     {user_readback.count():,}")

=== READ-BACK ROW COUNTS ===
Daily:    91
Content:  3,400
User:     24,999


In [0]:
# ============================================================
# WEEK 8 - READ-BACK SCHEMA CHECK
# ============================================================

print("=== DAILY READ-BACK SCHEMA ===")
daily_readback.printSchema()

print("=== CONTENT READ-BACK SCHEMA ===")
content_readback.printSchema()

print("=== USER READ-BACK SCHEMA ===")
user_readback.printSchema()

=== DAILY READ-BACK SCHEMA ===
root
 |-- session_date: date (nullable = true)
 |-- total_sessions: integer (nullable = true)
 |-- unique_users: integer (nullable = true)
 |-- unique_content_items: integer (nullable = true)
 |-- total_consumed_minutes: double (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: integer (nullable = true)
 |-- skipped_sessions: integer (nullable = true)
 |-- completion_rate_percent: double (nullable = true)

=== CONTENT READ-BACK SCHEMA ===
root
 |-- content_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- language: string (nullable = true)
 |-- is_platform_original: boolean (nullable = true)
 |-- total_sessions: integer (nullable = true)
 |-- unique_viewers: integer (nullable = true)
 |-- total_consumed_minutes: double (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: integer (nullable = true)
 |--

In [0]:
# ============================================================
# WEEK 8 - READ-BACK SCHEMA CHECK
# ============================================================

print("=== DAILY ===")
daily_readback.printSchema()

print("=== CONTENT ===")
content_readback.printSchema()

print("=== USER ===")
user_readback.printSchema()

=== DAILY ===
root
 |-- session_date: date (nullable = true)
 |-- total_sessions: integer (nullable = true)
 |-- unique_users: integer (nullable = true)
 |-- unique_content_items: integer (nullable = true)
 |-- total_consumed_minutes: double (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: integer (nullable = true)
 |-- skipped_sessions: integer (nullable = true)
 |-- completion_rate_percent: double (nullable = true)

=== CONTENT ===
root
 |-- content_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- language: string (nullable = true)
 |-- is_platform_original: boolean (nullable = true)
 |-- total_sessions: integer (nullable = true)
 |-- unique_viewers: integer (nullable = true)
 |-- total_consumed_minutes: double (nullable = true)
 |-- average_session_seconds: double (nullable = true)
 |-- completed_sessions: integer (nullable = true)
 |-- skipped_sessions: integer (nullab

In [0]:
# ============================================================
# WEEK 8 - READ-BACK KEY VALIDATION
# ============================================================

readback_checks = {
    "daily": ("session_date", daily_readback),
    "content": ("content_id", content_readback),
    "user": ("user_id", user_readback)
}

for name, (key, df) in readback_checks.items():
    rows = df.count()
    distinct_keys = df.select(key).distinct().count()
    null_keys = df.filter(df[key].isNull()).count()

    print(f"\n{name}")
    print(f"Rows: {rows:,}")
    print(f"Distinct keys: {distinct_keys:,}")
    print(f"Null keys: {null_keys:,}")
    print(
        "Key check:",
        "PASS" if rows == distinct_keys and null_keys == 0 else "FAIL"
    )


daily
Rows: 91
Distinct keys: 91
Null keys: 0
Key check: PASS

content
Rows: 3,400
Distinct keys: 3,400
Null keys: 0
Key check: PASS

user
Rows: 24,999
Distinct keys: 24,999
Null keys: 0
Key check: PASS


In [0]:
# ============================================================
# WEEK 8 - BUSINESS TOTAL RECONCILIATION
# ============================================================

print("=== GOLD TOTALS ===")

gold_totals = {}

for name, table in gold_tables.items():
    result = spark.sql(f"""
        SELECT
            SUM(total_sessions) AS total_sessions,
            SUM(total_consumed_minutes) AS total_consumed_minutes,
            SUM(completed_sessions) AS completed_sessions,
            SUM(skipped_sessions) AS skipped_sessions
        FROM {table}
    """).collect()[0]

    gold_totals[name] = result

    print(f"\n{name}")
    print(f"Sessions:   {result['total_sessions']}")
    print(f"Minutes:    {result['total_consumed_minutes']}")
    print(f"Completed:  {result['completed_sessions']}")
    print(f"Skipped:    {result['skipped_sessions']}")


print("\n=== READ-BACK TOTALS ===")

readbacks = {
    "daily_engagement": daily_readback,
    "content_performance": content_readback,
    "user_engagement": user_readback
}

for name, df in readbacks.items():
    result = df.select(
        "total_sessions",
        "total_consumed_minutes",
        "completed_sessions",
        "skipped_sessions"
    ).groupBy().sum().collect()[0]

    print(f"\n{name}")
    print(f"Sessions:   {result['sum(total_sessions)']}")
    print(f"Minutes:    {result['sum(total_consumed_minutes)']}")
    print(f"Completed:  {result['sum(completed_sessions)']}")
    print(f"Skipped:    {result['sum(skipped_sessions)']}")

=== GOLD TOTALS ===

daily_engagement
Sessions:   249701
Minutes:    6241936.65
Completed:  94689
Skipped:    47854

content_performance
Sessions:   249701
Minutes:    6241936.61
Completed:  94689
Skipped:    47854

user_engagement
Sessions:   249701
Minutes:    6241936.58
Completed:  94689
Skipped:    47854

=== READ-BACK TOTALS ===

daily_engagement
Sessions:   249701
Minutes:    6241936.650000003
Completed:  94689
Skipped:    47854

content_performance
Sessions:   249701
Minutes:    6241936.60999999
Completed:  94689
Skipped:    47854

user_engagement
Sessions:   249701
Minutes:    6241936.579999983
Completed:  94689
Skipped:    47854


In [0]:
# ============================================================
# WEEK 8 - DETERMINISTIC ORDERING CHECK
# ============================================================

print("=== FIRST / LAST KEYS ===")

print(
    "Daily:",
    daily_readback.select("session_date")
    .orderBy("session_date")
    .first()["session_date"],
    "->",
    daily_readback.select("session_date")
    .orderBy("session_date", ascending=False)
    .first()["session_date"]
)

print(
    "Content:",
    content_readback.select("content_id")
    .orderBy("content_id")
    .first()["content_id"],
    "->",
    content_readback.select("content_id")
    .orderBy("content_id", ascending=False)
    .first()["content_id"]
)

print(
    "User:",
    user_readback.select("user_id")
    .orderBy("user_id")
    .first()["user_id"],
    "->",
    user_readback.select("user_id")
    .orderBy("user_id", ascending=False)
    .first()["user_id"]
)

=== FIRST / LAST KEYS ===
Daily: 2025-12-20 -> 2026-03-31
Content: C000001 -> C_ORPHAN_249757
User: U000001 -> U_ORPHAN_249993


In [0]:
# ============================================================
# WEEK 8 - EXPORT READ-BACK SAMPLE
# ============================================================

print("=== DAILY EXPORT ===")
display(daily_readback.limit(5))

print("=== CONTENT EXPORT ===")
display(content_readback.limit(5))

print("=== USER EXPORT ===")
display(user_readback.limit(5))

=== DAILY EXPORT ===


session_date,total_sessions,unique_users,unique_content_items,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
2025-12-20,500,492,465,13006.6,1560.79,183,105,36.6
2026-01-01,2738,2586,1812,68765.95,1506.92,990,576,36.16
2026-01-02,2774,2633,1777,69452.48,1502.22,1072,538,38.64
2026-01-03,2790,2639,1791,68618.98,1475.68,1064,532,38.14
2026-01-04,2723,2571,1771,67240.62,1481.61,1008,520,37.02


=== CONTENT EXPORT ===


content_id,content_type,genre,language,is_platform_original,total_sessions,unique_viewers,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
C000001,VIDEO,Sports Stories,Telugu,false,98,98,2299.17,1407.65,33,27,33.67
C000002,MUSIC,Jazz,Telugu,false,89,89,174.67,117.75,26,21,29.21
C000003,MUSIC,Devotional,Hindi,false,75,75,201.02,160.81,30,11,40.0
C000004,MUSIC,Indie Pop,Kannada,true,82,81,126.97,92.9,23,24,28.05
C000005,PODCAST,Careers,Spanish,false,88,87,4383.18,2988.53,33,13,37.5


=== USER EXPORT ===


user_id,total_sessions,total_consumed_minutes,completed_sessions,skipped_sessions,average_session_seconds,first_session_date,last_session_date
U000001,6,34.03,2,1,340.33,2026-01-03,2026-03-30
U000002,10,240.52,4,1,1443.1,2026-01-01,2026-03-29
U000003,14,464.0,5,5,1988.57,2026-01-01,2026-03-25
U000004,7,127.37,2,1,1091.71,2026-01-03,2026-03-16
U000005,11,200.75,3,3,1095.0,2026-01-26,2026-03-27


In [0]:
%sql
SELECT
    SUM(total_sessions) AS total_sessions,
    SUM(total_consumed_minutes) AS total_consumed_minutes,
    SUM(completed_sessions) AS completed_sessions,
    SUM(skipped_sessions) AS skipped_sessions
FROM workspace.default.gold_daily_engagement;

total_sessions,total_consumed_minutes,completed_sessions,skipped_sessions
249701,6241936.65,94689,47854
